# 단어 분절 (BiLSTM 시퀀스 라벨링) — Disaster Tweets (PyTorch) — Colab

공백이 제거된 문자열에서 **단어 경계를 복원**하는 시퀀스 라벨링 기본 예제입니다.

- 핵심 흐름: 문장에서 공백 제거 → 각 문자마다 **다음에 경계가 오는가?**(이진 라벨) → **BiLSTM** 태깅 → 경계 예측.
- IOAI 2025 GAITE **Combinatorial Word Segmentation** 과제의 기반 기법입니다. (공식 해법도 LSTM 계열 시퀀스 라벨링)
- [NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started) 트윗을 코퍼스로 사용하며 토큰이 **노트북에 내장**되어 바로 실행됩니다.

> ⚙️ GPU 권장(작아서 CPU도 가능). ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부 공유 금지.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 트윗 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("nlp-getting-started", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 라벨 생성 (공백 제거 → 경계 라벨)

각 문장을 소문자 영문/숫자/공백만 남기고 단어들로 나눈 뒤, **공백 없는 문자열**과 각 문자 위치의 **경계 라벨**(이 문자가 한 단어의 마지막이면 1)을 만듭니다.

In [ ]:
import re, numpy as np, pandas as pd, torch

df = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
MAXLEN = 80

def make_example(text):
    text = re.sub(r"[^a-z0-9 ]", " ", str(text).lower())
    words = text.split()
    chars, labels = [], []
    for w in words:
        for i, ch in enumerate(w):
            chars.append(ch)
            labels.append(1 if i == len(w)-1 else 0)  # 단어 끝=경계
    return chars[:MAXLEN], labels[:MAXLEN]

examples = [make_example(t) for t in df["text"]]
examples = [(c,l) for c,l in examples if len(c) >= 5]

# 문자 사전
vocab = {"<pad>":0}
for c,_ in examples:
    for ch in c: vocab.setdefault(ch, len(vocab))
print("examples:", len(examples), "| vocab size:", len(vocab))

## 3. 텐서화 & DataLoader (패딩 + 마스크)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def encode(chars, labels):
    ids = [vocab[c] for c in chars]
    pad = MAXLEN - len(ids)
    mask = [1]*len(ids) + [0]*pad
    ids = ids + [0]*pad
    labels = labels + [0]*pad
    return ids, labels, mask

enc = [encode(c,l) for c,l in examples]
Xids = torch.tensor([e[0] for e in enc])
Y = torch.tensor([e[1] for e in enc], dtype=torch.float32)
M = torch.tensor([e[2] for e in enc], dtype=torch.float32)

n = len(Xids); cut = int(n*0.9)
g = torch.Generator().manual_seed(42); perm = torch.randperm(n, generator=g)
tr, va = perm[:cut], perm[cut:]
train_loader = DataLoader(TensorDataset(Xids[tr], Y[tr], M[tr]), batch_size=128, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| train:", len(tr), "val:", len(va))

## 4. BiLSTM 태거 & 학습

In [ ]:
import torch.nn as nn

class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, emb=64, hid=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hid*2, 1)
    def forward(self, x):
        h, _ = self.lstm(self.emb(x))
        return self.fc(h).squeeze(-1)  # (B, L) logits

model = BiLSTMTagger(len(vocab)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
lossf = nn.BCEWithLogitsLoss(reduction="none")

for epoch in range(1, 6):
    model.train(); tot=0; ntok=0
    for xb, yb, mb in train_loader:
        xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
        logit = model(xb)
        loss = (lossf(logit, yb) * mb).sum() / mb.sum()
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()*mb.sum().item(); ntok += mb.sum().item()
    print(f"epoch {epoch} | masked BCE {tot/ntok:.4f}")

## 5. 경계 예측 F1 평가

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
model.eval()
with torch.no_grad():
    logit = model(Xids[va].to(device)).cpu()
pred = (torch.sigmoid(logit) > 0.5).float()
mask = M[va].bool()
yt = Y[va][mask].numpy(); yp = pred[mask].numpy()
print(f"boundary  acc {accuracy_score(yt,yp):.4f} | F1 {f1_score(yt,yp):.4f}")

## 6. 분절 복원 데모

공백 없는 문자열을 넣어 단어 경계를 예측해 띄어쓰기를 복원합니다.

In [ ]:
def segment(s):
    s = re.sub(r"[^a-z0-9]", "", s.lower())[:MAXLEN]
    ids = torch.tensor([[vocab.get(c,0) for c in s]]).to(device)
    with torch.no_grad():
        p = torch.sigmoid(model(ids))[0].cpu().numpy()
    out = ""
    for ch, pr in zip(s, p):
        out += ch + (" " if pr > 0.5 else "")
    return out.strip()

for s in ["thefireisspreadingfast", "breakingnewsfloodwarning", "iloveplayingfootball"]:
    print(f"{s}  ->  {segment(s)}")

## 🏁 제출 안내

이 노트북은 **시퀀스 라벨링(단어 분절)** 연습용 데모입니다(제출 리더보드 없음).

- 코퍼스 출처: **[NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started)**

> IOAI 2025 GAITE *Combinatorial Word Segmentation* 과제의 기반 기법(시퀀스 라벨링/BiLSTM)입니다.